# P08 · Motor de costes operativos y ejecución
**QuantLab | Módulo 1 | Fundamentos 1.8 y 1.19**

**Pregunta:** ¿qué movimiento favorable necesita una estrategia para ganar dinero después de ejecutar sus órdenes?

Partimos del libro de órdenes (P02), las señales (P05) y el arbitraje (P06). Construiremos funciones de simulación spot sin apalancamiento ni órdenes reales. Primero estudia los ejemplos resueltos; después revisa los ejercicios resueltos de la sección 8. Los resultados sintéticos no son evidencia de rentabilidad en un mercado real.

Objetivos: separar comisiones, spread, profundidad, latencia/selección adversa y rotación; calcular el PnL exacto; identificar el umbral de equilibrio; comprobar que no se duplican costes.

Las funciones están en **`src/quantlab/p08/costs.py`** y sus pruebas en **`tests/test_costs.py`** (`pytest` desde la raíz del proyecto).

## 1. Configuración
Colab incluye las dependencias habituales. En local, instala `numpy pandas matplotlib requests jupyterlab` en tu entorno. Las comisiones son **supuestos didácticos**, no tarifas vigentes de ningún exchange. Todos los precios están en EUR/BTC, cantidades en BTC y comisiones en EUR (sin redondeos de exchange).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from pathlib import Path
from IPython.display import display
import inspect
import json
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p08.costs import (break_even, break_even_comparison, cost_attribution, fee_eur, fee_move_grid,
                                fetch_kraken_book, maker_vs_taker, manual_fee, markout, max_round_trips, round_trip,
                                size_table, synthetic_book, turnover_table, validate_book, walk_book)

MAKER_FEE_BPS = 5.0
TAKER_FEE_BPS = 10.0
QTY = 0.10
RUN_LIVE = False  # consulta pública opcional, no envía órdenes
USE_CSV = False   # libro L2 normalizado de tu laboratorio
BOOK_CSV = 'p08_book.csv'
EXPORT = False
plt.rcParams.update({'figure.figsize':(9,3.5),'axes.grid':True,'grid.alpha':.22,
                     'axes.spines.top':False,'axes.spines.right':False,'font.size':11})

## 2. Primer cálculo: puntos básicos y comisión
Un punto básico (bp) es 0,01 %. Una comisión de 10 bps equivale a 0,1 % del nominal.

$$f=\frac{bps}{10000},\qquad fee=q\,p_{exec}\, f$$

En una orden límite, **maker** solo si aporta liquidez y efectivamente se ejecuta. Una límite que cruza el libro generalmente es taker. No suponemos que poner una orden maker equivalga a ejecutarla.

In [ ]:
print(inspect.getsource(fee_eur))

example = pd.DataFrame({'type':['maker (si ejecutada)','taker'], 'fee_bps':[MAKER_FEE_BPS,TAKER_FEE_BPS]})
example['notional_eur'] = QTY*60000
example['fee_eur'] = [fee_eur(QTY,60000,b) for b in example.fee_bps]
display(example)

## 3. Simular la ejecución sobre el libro L2
Una compra consume asks, una venta consume bids. Si un nivel no basta, seguimos al siguiente. El precio medio es la suma del nominal ejecutado dividida por la cantidad ejecutada. No inventamos liquidez si la profundidad se agota.

El ejemplo representa una fotografía congelada del mercado. No incluye cambios del libro mientras viaja la orden, prioridad de cola ni reposición de liquidez. En vivo, la ejecución real puede diferir.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿el libro tiene sentido?</b>

`validate_book` se detiene con un error si hay lados distintos de `bid`/`ask`, niveles con precio o cantidad ≤ 0 o no finitos, o un libro incompleto (falta un lado), bloqueado (mejor bid = mejor ask) o cruzado (mejor bid > mejor ask). Los niveles con el mismo precio se suman.

El modo `RUN_LIVE` reutiliza la descarga del libro de P02 (`fetch_kraken_book`).

</div>

In [ ]:
print(inspect.getsource(validate_book))
print(inspect.getsource(synthetic_book))

book=synthetic_book()
provenance={'source':'synthetic','symbol':'BTC/EUR ficticio'}
if USE_CSV and RUN_LIVE: raise ValueError('Selecciona una única fuente para el libro')
if USE_CSV:
    book=validate_book(pd.read_csv(BOOK_CSV)); provenance={'source':BOOK_CSV,'symbol':'BTC/EUR (confirmar en metadatos)'}
elif RUN_LIVE:
    book,provenance=fetch_kraken_book()  # si falla, lanza el error
print(provenance)
display(book.sort_values(['side','price']).head(8))

### 3.1 Motor de ejecución y desglose del coste
Sea $s=+1$ en compra y $s=-1$ en venta. Usamos el mid contemporáneo $m$ y el mejor precio del lado consumido $b$.

$$C_{spread}=s q(b-m),\qquad C_{depth}=s q(p_{exec}-b)$$
$$C_{exec}=C_{spread}+C_{depth}+fee$$

El shortfall respecto a un mid de decisión distinto incluye además el movimiento entre decisión y llegada. Para una ejecución parcial mostramos el coste sobre lo ejecutado, no el coste de oportunidad de lo que quedó pendiente.

In [ ]:
print(inspect.getsource(walk_book))

fills = size_table(book, [.01,.1,.5,1.,3.], TAKER_FEE_BPS)
display(fills[['side','requested_qty','filled_qty','fill_ratio','execution_price','fee','spread_cost','slippage_cost','execution_cost_bps']].round(4))

In [ ]:
fig,ax=plt.subplots()
for side,group in fills.groupby('side'):
    complete=group[np.isclose(group.fill_ratio,1)]
    ax.plot(complete.requested_qty,complete.execution_cost_bps,marker='o',label=side)
    partial=group[~np.isclose(group.fill_ratio,1)]
    ax.scatter(partial.requested_qty,partial.execution_cost_bps,marker='x',s=80,label=f'{side}: parcial' if len(partial) else '_nolegend_')
ax.set(title='Coste de ejecución por tamaño solicitado',xlabel='Cantidad solicitada (BTC)',ylabel='Coste sobre cantidad ejecutada (bps)');ax.legend();plt.show()

**Interpretación:** una orden grande puede consumir niveles peores y quedar incompleta. Una cruz representa una ejecución parcial, no una oportunidad ejecutable por todo el tamaño. El coste por unidad puede aumentar aunque la comisión porcentual sea constante.

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · La comisión domina en órdenes pequeñas; la profundidad, en las grandes</b>

Compras con 10 bps de comisión (las ventas son casi simétricas):

| Tamaño | Ejecutado | Niveles | Comisión | Spread | Profundidad | Coste total |
|---|---|---|---|---|---|---|
| 0,01 BTC | 100 % | 1 | 0,60 € | 0,03 € | 0,00 € | **10,50 bps** |
| 0,10 BTC | 100 % | 2 | 6,00 € | 0,30 € | 0,21 € | 10,85 bps |
| 0,50 BTC | 100 % | 4 | 30,01 € | 1,50 € | 6,81 € | 12,77 bps |
| 1,00 BTC | 100 % | 5 | 60,03 € | 3,00 € | 29,31 € | 15,39 bps |
| 3,00 BTC | **67 %** | 6 | 120,13 € | 6,00 € | **122,31 €** | 20,70 bps |

- En **0,01 BTC**, la comisión es el 95 % del coste: el spread (3 € de medio spread) apenas cuenta y no hay profundidad.
- A medida que crece el tamaño, la **profundidad** sube mucho más rápido que la comisión: con 3 BTC ya supera a la comisión.
- La orden de **3 BTC queda incompleta**: el libro solo tiene 2 BTC por lado. Su coste en bps es sobre lo ejecutado, no sobre lo pedido.

</div>

## 4. Ida y vuelta: PnL exacto y umbral de equilibrio
Compra en $p_b$ y venta de la misma cantidad en $p_s$, con comisiones respectivas $f_b,f_s$ en decimales:

$$PnL=q[p_s(1-f_s)-p_b(1+f_b)]$$
$$p_s^*=p_b\frac{1+f_b}{1-f_s}$$

Este umbral es para el **precio ejecutado** de venta, no el mid. Si calculamos PnL usando precios de ejecución, no volvemos a restar spread ni slippage. Para obtener el umbral en mid añadimos las condiciones de ejecución previstas. Modelaremos dos libros sintéticos separados por un desplazamiento del mid; no son un histórico real ni una simulación dinámica de impacto.

In [ ]:
print(inspect.getsource(round_trip))
print(inspect.getsource(break_even))

base=synthetic_book(); plus=synthetic_book(60000*1.0035)
trade=round_trip(base,plus,QTY,TAKER_FEE_BPS,TAKER_FEE_BPS)
display(pd.Series(trade,name='valor').round(5).to_frame())
be = break_even(synthetic_book, QTY, TAKER_FEE_BPS, TAKER_FEE_BPS)
break_even_exec, break_even_mid, break_even_bps = be['break_even_exec_price'], be['break_even_mid'], be['break_even_move_bps']
print(f'Umbral de venta ejecutada: {break_even_exec:,.2f} EUR/BTC | Movimiento mid: {break_even_bps:.3f} bps')

In [ ]:
fig,ax=plt.subplots()
labels=['Bruto','Spread','Profundidad','Comisiones','Neto']
values=[trade['gross_pnl'],-trade['spread_cost'],-trade['slippage_cost'],-trade['fees'],trade['net_pnl']]
ax.bar(labels,values,color=['#457b9d' if x>=0 else '#b65f52' for x in values]);ax.axhline(0,color='black',lw=.8)
ax.set(title='Descomposición del PnL · movimiento mid de 35 bps',ylabel='EUR por operación');plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Con un movimiento de 35 bps se gana 8 €; hacen falta 21,7 bps solo para empatar</b>

| Concepto | EUR |
|---|---|
| Bruto (mid +35 bps × 0,1 BTC) | +21,00 |
| Spread (las dos piernas) | −0,60 |
| Profundidad (las dos piernas) | −0,39 |
| Comisiones (10 bps por lado) | −12,02 |
| **Neto** | **+7,99** (+13,3 bps) |

- **Conciliación:** 21,00 − 0,60 − 0,39 − 12,02 = 7,99. El neto se calcula con los precios **ejecutados**, así que no se vuelven a restar spread ni profundidad.
- **Umbral de equilibrio:** hay que vender a **60.125,23 €** (precio ejecutado). Como cada venta de 0,1 BTC se ejecuta 4,8 € por debajo del mid, el mid tiene que subir hasta 60.130,03 €: un movimiento de **+21,67 bps** para quedar en cero.

</div>

### 4.1 Sensibilidad a comisiones
Comparamos exclusivamente tarifas hipotéticas de órdenes **taker** manteniendo el mismo libro. Un escenario de comisión baja no representa por sí mismo ejecución maker.

In [ ]:
print(inspect.getsource(fee_move_grid))

sensitivity = fee_move_grid(synthetic_book, QTY, [0,5,10,25,40], np.arange(-20,101,5))
fig,ax=plt.subplots()
for fee,g in sensitivity.groupby('fee_bps'):
    ax.plot(g.move_bps,g.net_pnl,label=f'{fee} bps/lado')
ax.axhline(0,color='black',lw=.8);ax.set(title='Rentabilidad neta según movimiento y comisión',xlabel='Movimiento del mid (bps)',ylabel='PnL neto (EUR)');ax.legend(ncol=2);plt.show()
display(sensitivity[sensitivity.move_bps.isin([0,20,35,50])].pivot(index='move_bps',columns='fee_bps',values='net_pnl').round(2))

## 5. Maker, Taker y selección adversa
Una comisión maker reducida puede ser compensada por el riesgo de no ejecución, posición en cola y selección adversa. Definimos *markout* neto de comisión (sin cierre) a un horizonte $h$:

$$markout_h=sq(m_{t+h}-p_{fill})-fee$$

Un markout negativo significa que el precio posterior, valorado al mid, ha resultado desfavorable después de comisión. No es PnL de una posición ya cerrada. La siguiente compra límite es una ejecución hipotética condicionada a haber sido rellenada; no simulamos la probabilidad de fill.

In [ ]:
print(inspect.getsource(markout))

maker_price=59997
maker_cases=pd.DataFrame({'future_mid':[59970,59997,60000,60030]})
maker_cases['markout_eur']=[markout('buy',QTY,maker_price,m,MAKER_FEE_BPS) for m in maker_cases.future_mid]
display(maker_cases.round(4))

## 6. Rotación y erosión del beneficio
Dos estrategias pueden tener el mismo retorno bruto y costes muy diferentes. Simulamos escenarios de operaciones secuenciales de tamaño fijo (sin reinversión ni solapamiento), capital de referencia de 10.000 EUR y nominal de compra teórico de 1.000 EUR por operación. El coste de ida y vuelta de 25 bps es aproximado sobre el nominal inicial, no un cálculo exacto de fills. El turnover de dos lados es un proxy que supone nominal similar en compra y venta.

In [ ]:
print(inspect.getsource(turnover_table))

capital=10000.; nominal=1000.; gross_total=100.; cost_bps=25.
turnover = turnover_table([10,20,40,80,120], capital, nominal, gross_total, cost_bps)
display(turnover)
fig,ax=plt.subplots();ax.plot(turnover.round_trips,turnover.net_pnl,marker='o');ax.axhline(0,color='black',lw=.8)
ax.set(title='Mismo beneficio bruto, distinta rotación',xlabel='Operaciones completas',ylabel='PnL neto (EUR)');plt.show()

## 7. Comprobaciones automáticas
Validamos con identidades contables y casos controlados antes de confiar en el motor. No validan la capacidad predictiva ni la calidad de una ejecución real.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: identidades contables y casos controlados
checks=[]
def check(label,condition):
    assert condition,label
    checks.append(label)
test_buy=walk_book(base,'buy',.1,10)
check('Conversión de comisión',np.isclose(fee_eur(.1,60000,10),6))
check('Precio medio ponderado',np.isclose(test_buy['execution_price'],60005.1))
check('Spread compra',np.isclose(test_buy['spread_cost'],.3))
check('Slippage compra',np.isclose(test_buy['slippage_cost'],.21))
check('Fill parcial',np.isclose(walk_book(base,'buy',4)['filled_qty'],2))
check('Coste igual a shortfall sin retraso',np.isclose(test_buy['execution_cost'],test_buy['implementation_shortfall']))
check('Coste de retraso',np.isclose(walk_book(base,'buy',.1,10,59990)['delay_cost'],1))
check('Conciliación PnL',np.isclose(trade['gross_pnl']-trade['total_cost'],trade['net_pnl']))
check('Equilibrio exacto',np.isclose(round_trip(base,synthetic_book(break_even_mid),QTY,TAKER_FEE_BPS,TAKER_FEE_BPS)['net_pnl'],0))
check('Mayor tarifa reduce PnL',round_trip(base,plus,QTY,20,20)['net_pnl']<round_trip(base,plus,QTY,10,10)['net_pnl'])
try:
    round_trip(base,plus,4,10,10)
except ValueError: checks.append('Rechaza round trip incompleto')
else: raise AssertionError('No rechazó posición incompleta')
print(f'{len(checks)} comprobaciones superadas:');print(' · '.join(checks))

## 8. Ejercicios
**E1.** Implementa `fee_eur` por tu cuenta. Verifica que 30 bps sobre 2.000 EUR son 6 EUR.

**E2.** Calcula a mano la compra de 0,10 BTC del libro sintético. Se consumen dos niveles. Comprueba precio medio, comisión y descomposición de costes.

**E3.** Simula cantidades 0,01, 0,1, 0,5, 1 y 3 BTC. ¿Cuándo se agota el mejor precio? ¿Cuándo queda una orden incompleta? Explica por qué no es válido simular su cierre íntegro.

**E4.** Calcula el umbral de equilibrio para una comisión taker de 5, 10 y 25 bps por lado. Usa la fórmula exacta y contrasta con la aproximación de sumar costes en bps.

**E5.** Compara la compra maker hipotética con la taker cuando el mid futuro es 59.970, 60.000 y 60.030 EUR. ¿Qué datos necesitarías para comparar la rentabilidad no condicionada al fill?

**E6.** Calcula cuántas operaciones puede soportar el escenario de rotación antes de perder dinero. Contrasta 10, 25 y 50 bps de coste total.

**E7.** Activa `RUN_LIVE=True`, vuelve a ejecutar desde arriba y analiza las compras y ventas de tamaño pequeño. Utiliza un único snapshot: no interpretes la ida y vuelta sintética como una operación real.

**E8.** Diseña una interfaz de costes para P05/P06 que reciba precios de decisión, fills, comisiones y timestamps. Identifica dónde podría duplicarse el coste y cómo detectarías una cotización obsoleta.

### E1 · Comisión propia

In [ ]:
print(inspect.getsource(manual_fee))

e1 = manual_fee(2000, 30)
print(f"30 bps sobre 2.000 EUR = {e1:.2f} EUR")
assert np.isclose(e1, 6.0)
assert np.isclose(manual_fee(QTY * 60000, TAKER_FEE_BPS), fee_eur(QTY, 60000, TAKER_FEE_BPS))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Comisión</b>

- 2.000 × 30 / 10.000 = **6 EUR**. Es lo mismo que 0,30 % del nominal.
- `fee_eur(q, p, bps)` y `manual_fee(nominal, bps)` son la misma fórmula: la primera construye el nominal como q × p. Para 0,1 BTC a 60.000 € con 10 bps, las dos dan 6 €.

</div>

### E2 · Compra de 0,10 BTC a mano

In [ ]:
asks = book[book.side == 'ask'].sort_values('price').head(2)
display(asks)
by_hand_price = (0.03 * 60003 + 0.07 * 60006) / 0.10
r = walk_book(base, 'buy', 0.10, TAKER_FEE_BPS)
e2 = pd.DataFrame({'a mano': [by_hand_price, 0.10 * by_hand_price * 0.001, 0.10 * (60003 - 60000), 0.10 * (by_hand_price - 60003)],
                   'walk_book': [r['execution_price'], r['fee'], r['spread_cost'], r['slippage_cost']]},
                  index=['precio medio', 'comisión', 'spread', 'profundidad'])
display(e2.round(5))
assert np.allclose(e2['a mano'], e2['walk_book'])

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Compra de 0,10 BTC, paso a paso</b>

| Nivel | Precio | Disponible | Tomado | Nominal |
|---|---|---|---|---|
| 1 | 60.003 € | 0,03 BTC | 0,03 BTC | 1.800,09 € |
| 2 | 60.006 € | 0,07 BTC | 0,07 BTC | 4.200,42 € |
| **Total** | | | **0,10 BTC** | **6.000,51 €** |

- **Precio medio** = 6.000,51 / 0,10 = **60.005,10 €**.
- **Comisión** = 6.000,51 × 0,001 = **6,0005 €**.
- **Spread** = 0,10 × (60.003 − 60.000) = **0,30 €**: el mejor ask está 3 € por encima del mid.
- **Profundidad** = 0,10 × (60.005,10 − 60.003) = **0,21 €**: el segundo nivel es 3 € peor y aporta el 70 % de la cantidad.
- **Coste total** = 0,30 + 0,21 + 6,0005 = **6,51 €** (10,85 bps). Coincide con `walk_book`.

</div>

### E3 · Tamaños y profundidad

In [ ]:
e3 = size_table(book, [.01, .03, .04, .1, .5, 1., 2., 3.], TAKER_FEE_BPS)
display(e3[['side', 'requested_qty', 'filled_qty', 'fill_ratio', 'levels_used', 'execution_price', 'execution_cost_bps']].round(4))
print("Cantidad total por lado:", book.groupby('side').quantity.sum().to_dict())

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · El mejor precio dura 0,03 BTC; la profundidad, 2 BTC</b>

- **El mejor precio se agota enseguida:** el mejor ask solo tiene **0,03 BTC** y el mejor bid, **0,04 BTC**. Una compra de 0,1 BTC ya usa dos niveles.
- **Orden incompleta:** cada lado tiene **2 BTC** en total. Hasta 2 BTC se ejecuta entera; la de 3 BTC solo ejecuta 2 BTC (67 %).
- **¿Por qué no se puede simular su cierre íntegro?** Porque solo se compraron 2 BTC: no existe una posición de 3 BTC que cerrar. Suponer que el tercer BTC se compró inventaría liquidez que el snapshot no muestra, y suponer que se cierra todo calcularía un PnL de una operación que nunca ocurrió. Por eso `round_trip` **rechaza** las piernas incompletas. Para simularlo de verdad haría falta un libro más profundo o modelar cómo se repone la liquidez con el tiempo.

</div>

### E4 · Umbral exacto frente a la suma de costes

In [ ]:
print(inspect.getsource(break_even_comparison))

e4 = break_even_comparison(synthetic_book, QTY, [5, 10, 25])
display(e4.round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · Sumar bps es una buena aproximación, pero siempre se queda corta</b>

0,1 BTC; la aproximación suma dos comisiones más el spread y la profundidad de las dos piernas (1,65 bps):

| Comisión por lado | Umbral exacto | Aproximación | Diferencia |
|---|---|---|---|
| 5 bps | 11,656 bps | 11,65 bps | +0,006 bps |
| 10 bps | 21,672 bps | 21,65 bps | +0,022 bps |
| 25 bps | 51,780 bps | 51,65 bps | +0,130 bps |

- La fórmula exacta es multiplicativa, (1 + f_b) / (1 − f_s); la suma ignora los términos cruzados (f_b · f_s, y que la comisión de venta se aplica sobre un precio más alto).
- La aproximación **subestima siempre** el umbral, y el error crece con la comisión: es despreciable con 5 bps, pero con comisiones altas o muchas operaciones conviene usar la exacta.

</div>

### E5 · Maker frente a taker

In [ ]:
print(inspect.getsource(maker_vs_taker))

e5 = maker_vs_taker(base, QTY, 59997, [59970, 60000, 60030], MAKER_FEE_BPS, TAKER_FEE_BPS)
print("Precio taker:", e5.attrs['taker_price'], "| precio maker:", e5.attrs['maker_price'])
display(e5.round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · Condicionado al fill, la orden maker siempre parece mejor</b>

| Mid futuro | Markout maker (59.997 €, 5 bps) | Markout taker (60.005,10 €, 10 bps) | Ventaja maker |
|---|---|---|---|
| 59.970 € | −5,70 € | −9,51 € | +3,81 € |
| 60.000 € | −2,70 € | −6,51 € | +3,81 € |
| 60.030 € | +0,30 € | −3,51 € | +3,81 € |

- **Si la orden maker se rellena**, gana a la taker por la misma cantidad en todos los casos (**3,81 €**): compra 8,1 € más barato por BTC y paga la mitad de comisión.
- **Pero esa comparación está sesgada.** Una orden límite de compra suele rellenarse cuando el precio **cae** hacia ella (selección adversa), y **no** se rellena cuando el precio sube, justo cuando la taker más gana. Comparar solo los casos rellenados oculta esos dos efectos.

**Datos necesarios para una comparación no condicionada al fill:**
- **Probabilidad de fill** según la distancia al mid, el tamaño y la **posición en la cola** (cuánto volumen había delante a ese precio).
- **Tiempo hasta el fill** y si fue parcial.
- **Markouts de los casos sin fill:** qué hizo el precio mientras la orden esperaba. Si no se rellena, o se pierde la oportunidad o se termina ejecutando como taker a un precio peor.
- **Trades y actualizaciones del libro con marcas de tiempo** (P02, P03), para reconstruir la cola y saber quién ejecutó contra la orden.

</div>

### E6 · Operaciones máximas antes de perder

In [ ]:
print(inspect.getsource(max_round_trips))

e6 = max_round_trips(gross_total, nominal, [10, 25, 50])
display(e6)

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Cuantas más operaciones, menos margen</b>

Beneficio bruto total: 100 €; nominal por operación: 1.000 €.

| Coste por ida y vuelta | Coste por operación | Operaciones para empatar | Máximo con beneficio |
|---|---|---|---|
| 10 bps | 1,00 € | 100 | **99** |
| 25 bps | 2,50 € | 40 | **39** |
| 50 bps | 5,00 € | 20 | **19** |

- El número de operaciones que se pueden pagar es **bruto / coste por operación**. Con el mismo beneficio bruto, duplicar el coste reduce a la mitad las operaciones posibles.
- Con 25 bps, la tabla de rotación lo muestra: 40 operaciones dejan 0 €, y 120 operaciones (turnover de 24 veces el capital) pierden 200 €.

</div>

### E7 · Libro real de Kraken
Cambia `LIVE_E7` a `True` para descargar un único snapshot del libro y analizar compras y ventas pequeñas. Si la conexión falla, la celda se detiene con el error.

In [ ]:
LIVE_E7 = False

if LIVE_E7 or RUN_LIVE:
    live_book, live_prov = (book, provenance) if RUN_LIVE else fetch_kraken_book()
    print(live_prov)
    live_bid = live_book[live_book.side == 'bid'].price.max(); live_ask = live_book[live_book.side == 'ask'].price.min()
    print(f"Mejor bid {live_bid:,.1f} | mejor ask {live_ask:,.1f} | spread {live_ask - live_bid:.2f} EUR "
          f"({(live_ask - live_bid) / ((live_ask + live_bid) / 2) * 1e4:.3f} bps)")
    print("Cantidad por lado en el snapshot (BTC):", live_book.groupby('side').quantity.sum().round(3).to_dict())
    e7 = size_table(live_book, [.001, .01, .1, .5, 1.], TAKER_FEE_BPS)
    display(e7[['side', 'requested_qty', 'filled_qty', 'levels_used', 'execution_price', 'fee', 'spread_cost', 'slippage_cost', 'execution_cost_bps']].round(4))
else:
    print("LIVE_E7 = False: no se consulta Kraken. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Kraken BTC/EUR: en órdenes pequeñas solo cuenta la comisión</b>

**Snapshot:** 19-11-2026, recepción 20:41:18 UTC, 50 niveles por lado. Mejor bid 56.566,0 € (0,242 BTC), mejor ask 56.566,3 € (0,242 BTC). **Spread: 0,30 € (0,053 bps).** Cantidad visible: 20,9 BTC en bids y 13,3 BTC en asks.

| Tamaño | Niveles (compra / venta) | Comisión | Spread | Profundidad (compra / venta) | Coste total (compra / venta) |
|---|---|---|---|---|---|
| 0,001 BTC | 1 / 1 | 0,06 € | 0,0002 € | 0 / 0 | 10,03 / 10,03 bps |
| 0,01 BTC | 1 / 1 | 0,57 € | 0,0015 € | 0 / 0 | 10,03 / 10,03 bps |
| 0,1 BTC | 1 / 1 | 5,66 € | 0,015 € | 0 / 0 | 10,03 / 10,03 bps |
| 0,5 BTC | 2 / 3 | 28,28 € | 0,075 € | 0,59 / 0,58 € | 10,24 / 10,23 bps |
| 1 BTC | 4 / 7 | 56,57 € | 0,15 € | 1,87 / 3,39 € | 10,36 / 10,62 bps |

- **Hasta 0,1 BTC, la comisión es el 99,7 % del coste.** El mejor nivel tenía 0,242 BTC, así que no hay profundidad que pagar, y el medio spread es de solo 0,15 € por BTC.
- Incluso con 1 BTC, la profundidad añade menos de 0,6 bps a los 10 bps de comisión. La venta recorre más niveles (7 frente a 4) porque los bids cercanos tenían menos cantidad.
- **Una ida y vuelta de 0,01 BTC sobre este mismo snapshot costaría 20,05 bps**: casi exactamente dos comisiones. Pero es un único snapshot: no es una operación real, el libro cambiaría entre la compra y la venta, y no hay información sobre la cola ni sobre la reposición de liquidez.
- La comisión real de Kraken para un particular es mayor que este supuesto (0,80 % taker en el nivel 1 de Kraken Pro, consultado en P01), lo que hace todavía más dominante la comisión.

</div>

### E8 · Interfaz de costes para P05/P06

In [ ]:
print(inspect.getsource(cost_attribution))

t0 = pd.Timestamp("2026-01-01T10:00:00Z")
e8_input = pd.DataFrame([
    dict(fill_id="a", side="buy", quantity=.1, decision_mid=60000, decision_at=t0, quote_mid=60002,
         quote_at=t0 - pd.Timedelta("0.5s"), best_price=60005, fill_price=60008, fill_at=t0 + pd.Timedelta("1s"), fee_eur=6.0),
    dict(fill_id="b", side="sell", quantity=.1, decision_mid=60000, decision_at=t0, quote_mid=59990,
         quote_at=t0 - pd.Timedelta("5s"), best_price=59987, fill_price=59985, fill_at=t0 + pd.Timedelta("1s"), fee_eur=6.0),
    dict(fill_id="b", side="sell", quantity=.1, decision_mid=60000, decision_at=t0, quote_mid=59990,
         quote_at=t0, best_price=59987, fill_price=59985, fill_at=t0 - pd.Timedelta("1s"), fee_eur=6.0),
    dict(fill_id="c", side="buy", quantity=.1, decision_mid=60000, decision_at=t0, quote_mid=60002,
         quote_at=t0 + pd.Timedelta("0.2s"), best_price=60005, fill_price=60005, fill_at=t0 + pd.Timedelta("1s"), fee_eur=6.0),
])
e8 = cost_attribution(e8_input)
display(e8[['fill_id', 'side', 'delay_cost', 'spread_cost', 'slippage_cost', 'fee_eur', 'implementation_shortfall',
            'reconciles', 'quote_age', 'stale_quote', 'future_quote', 'fill_before_decision', 'duplicated_fill']]
        .round({c: 3 for c in ['delay_cost', 'spread_cost', 'slippage_cost', 'implementation_shortfall']}))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Una interfaz en la que cada coste se cuenta una sola vez</b>

**Entrada (una fila por fill):** `fill_id`, `side`, `quantity`, `decision_mid` y `decision_at` (lo que vio la estrategia al decidir), `quote_mid` y `quote_at` (la cotización contra la que se envía la orden), `best_price` (mejor precio del lado consumido), `fill_price` y `fill_at` (ejecución real), y `fee_eur` (comisión real del exchange, no un porcentaje supuesto).

**Descomposición en tramos que no se solapan:**

| Tramo | Fórmula | Qué mide |
|---|---|---|
| Retraso | s · q · (quote_mid − decision_mid) | Lo que se movió el mercado entre decidir y enviar |
| Spread | s · q · (best_price − quote_mid) | Cruzar del mid al mejor precio |
| Profundidad | s · q · (fill_price − best_price) | Recorrer niveles peores |
| Comisión | fee_eur | Lo cobrado por el exchange |

Su suma es el *implementation shortfall*, y debe ser **exactamente** s · q · (fill_price − decision_mid) + fee. La columna `reconciles` lo comprueba en cada fila.

**Dónde podría duplicarse el coste:**
- Restar el spread y la profundidad **además** de usar precios ejecutados en el PnL (P08, sección 4).
- Sumar una comisión porcentual supuesta cuando `fill_price` ya es neto de comisión, o contar dos veces el mismo fill (`duplicated_fill`, ejemplo "b").
- En P06, restar el spread a una ruta que ya compra al ask y vende al bid.
- En P05, aplicar un coste fijo por ciclo y además los costes del libro.

**Cómo detectar problemas de tiempo:**
- `stale_quote`: la cotización tenía más de 2 s al decidir (ejemplo "b": 5 s). La decisión se basó en un precio viejo.
- `future_quote`: la cotización es **posterior** a la decisión (ejemplo "c"). Es *look-ahead*: en tiempo real no podía conocerse.
- `fill_before_decision`: un fill anterior a la decisión (segunda fila "b") indica relojes desincronizados o un emparejamiento erróneo de fills y órdenes.

</div>

## 9. Exportación e integración
El CSV de libro debe tener `side` (bid/ask), `price` (EUR/BTC), `quantity` (BTC), todos del mismo snapshot y mercado. No mezcles snapshots o exchanges. Los resultados de ejecución pueden usarse en P09; el código ya está en `src/quantlab/p08/costs.py`, con sus pruebas en `tests/test_costs.py`.

Fuente pública opcional: [Kraken Get Order Book](https://docs.kraken.com/api-reference/market-data/get-order-book). El endpoint L2 agrega cantidades por nivel. Guarda la procedencia y la hora de recepción: no equivalen a una garantía de ejecución. No hay credenciales ni órdenes reales.

In [ ]:
if EXPORT:
    out=Path('quantlab_exports')/f"p08_{pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ')}"
    out.mkdir(parents=True,exist_ok=True)
    book.to_csv(out/'book.csv',index=False);fills.to_csv(out/'execution_costs.csv',index=False)
    sensitivity.to_csv(out/'synthetic_sensitivity.csv',index=False)
    (out/'metadata.json').write_text(json.dumps({**provenance,'fee_bps_hypothetical':TAKER_FEE_BPS,'quantity_btc':QTY,'round_trip_source':'synthetic'},indent=2),encoding='utf-8')
    print(out)
else: print('Exportación desactivada. Activa EXPORT si quieres guardar los resultados.')

## 10. Conclusiones y próxima práctica

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P08</b>

**¿Qué coste domina para órdenes pequeñas?** La **comisión**. En el libro sintético es el 95 % del coste de 0,01 BTC; en el libro real de Kraken, el 99,7 % hasta 0,1 BTC, porque el spread era de 0,05 bps y el mejor nivel tenía 0,242 BTC.

**¿Y para las grandes?** La **profundidad** (slippage). En el libro sintético pasa de 0,21 € con 0,1 BTC a 122 € con 2 BTC, y supera a la comisión. Además, una orden más grande que el libro queda **incompleta** (3 BTC solo ejecuta 2).

**¿Qué movimiento necesitas para cubrir gastos?** Con 10 bps por lado y 0,1 BTC, el mid tiene que subir **+21,67 bps** solo para empatar: dos comisiones (≈ 20 bps) más spread y profundidad (≈ 1,65 bps). Con 25 bps por lado, +51,8 bps. Sumar bps da una buena aproximación, pero siempre se queda un poco corta.

**¿Qué supuestos no representan la ejecución real?**
- El libro es una **foto congelada**: no cambia mientras viaja la orden, no hay latencia, ni cola, ni reposición de liquidez, ni otros participantes reaccionando.
- La orden maker se supone **rellenada**: no se modela la probabilidad de fill ni la selección adversa de los casos no rellenados.
- La ida y vuelta usa un segundo libro que es el primero **desplazado**: no un mercado real, ni el impacto de la propia compra.
- Las comisiones son supuestos fijos, sin niveles por volumen, sin mínimos ni redondeos.
- La rotación usa un coste aproximado sobre el nominal, no fills reales.

**Recordatorio:** un resultado positivo en este ejercicio no demuestra alpha. En P09 evaluaremos series de PnL, esperanza matemática, Sharpe, drawdown y profit factor.

📄 Resumen en `docs/conclusiones/P08_transaction_costs.md`

</div>